In [128]:
import pandas as pd
import numpy as np
from texas_gerrymandering_hb4.config import GEO_VTD

In [129]:
geo = pd.read_parquet(GEO_VTD)
geo.shape, geo.columns.tolist()[:50]


((9712, 25),
 ['vtd_geoid',
  'total_pop',
  'total_nh_white',
  'total_nh_black',
  'total_hisp',
  'total_nh_asian',
  'total_nh_native',
  'total_nh_pi',
  'total_other',
  'vap_total',
  'vap_nh_white',
  'vap_nh_black',
  'vap_hisp',
  'vap_nh_asian',
  'vap_nh_native',
  'vap_other',
  'cvap_total',
  'cvap_hisp',
  'cvap_nh_white',
  'cvap_nh_black',
  'cvap_nh_asian',
  'cvap_nh_native',
  'cvap_nh_pi',
  'cvap_other',
  'state_fips'])

In [130]:
required = [
    "vtd_geoid", "total_pop",
    "vap_total", "vap_hisp", "vap_nh_white", "vap_nh_black", "vap_other",
    "cvap_total", "cvap_hisp", "cvap_nh_white", "cvap_nh_black", "cvap_other",
    "total_hisp", "total_nh_white", "total_nh_black", "total_other"
]
missing = [c for c in required if c not in geo.columns]
missing


[]

In [131]:
num_cols = [c for c in geo.columns if c not in ["vtd_geoid", "state_fips"]]

neg = (geo[num_cols] < 0).sum().sort_values(ascending=False)
neg[neg > 0].head(20)


Series([], dtype: int64)

In [132]:
bad_cvap_vap = (geo["cvap_total"] > geo["vap_total"]).sum()
bad_vap_tot  = (geo["vap_total"] > geo["total_pop"]).sum()

bad_cvap_vap, bad_vap_tot, len(geo)


(np.int64(2899), np.int64(0), 9712)

In [133]:
geo.assign(diff_cvap_vap=geo["cvap_total"]-geo["vap_total"])\
   .sort_values("diff_cvap_vap", ascending=False)\
   .head(20)[["vtd_geoid","cvap_total","vap_total","diff_cvap_vap"]]


,vtd_geoid,cvap_total,vap_total,diff_cvap_vap
3630,3631.0,15956,11773,4183
1824,1825.0,6393,2761,3632
6853,6854.0,6756,4089,2667
2993,2994.0,5591,2944,2647
6583,6584.0,2975,375,2600
5202,5203.0,6013,3421,2592
949,950.0,3905,1408,2497
1654,1655.0,6788,4514,2274
3659,3660.0,13335,11084,2251
1169,1170.0,7544,5346,2198


In [134]:
vap_groups = ["vap_hisp","vap_nh_white","vap_nh_black","vap_nh_asian","vap_nh_native","vap_other"]
cvap_groups = ["cvap_hisp","cvap_nh_white","cvap_nh_black","cvap_nh_asian","cvap_nh_native","cvap_nh_pi","cvap_other"]
tot_groups = ["total_hisp","total_nh_white","total_nh_black","total_nh_asian","total_nh_native","total_nh_pi","total_other"]

# only keep columns that exist
vap_groups = [c for c in vap_groups if c in geo.columns]
cvap_groups = [c for c in cvap_groups if c in geo.columns]
tot_groups = [c for c in tot_groups if c in geo.columns]

bad_vap = (geo[vap_groups].sum(axis=1) > geo["vap_total"]).sum()
bad_cvap = (geo[cvap_groups].sum(axis=1) > geo["cvap_total"]).sum()
bad_tot = (geo[tot_groups].sum(axis=1) > geo["total_pop"]).sum()

bad_vap, bad_cvap, bad_tot


(np.int64(1356), np.int64(225), np.int64(2319))

In [135]:
tot_total = int(geo["total_pop"].sum())
vap_total = int(geo["vap_total"].sum())
cvap_total = int(geo["cvap_total"].sum())

tot_total, vap_total, cvap_total


(29145505, 21866700, 19868069)

In [136]:
def pct(x): return f"{x*100:.2f}%"

table = pd.DataFrame([
    ("Latino", geo["total_hisp"].sum(), geo["vap_hisp"].sum(), geo["cvap_hisp"].sum()),
    ("Black",  geo["total_nh_black"].sum(), geo["vap_nh_black"].sum(), geo["cvap_nh_black"].sum()),
    ("White",  geo["total_nh_white"].sum(), geo["vap_nh_white"].sum(), geo["cvap_nh_white"].sum()),
    ("Other",  geo["total_other"].sum(), geo["vap_other"].sum(), geo["cvap_other"].sum()),
], columns=["group","total_count","vap_count","cvap_count"])

table["share_total"] = table["total_count"]/tot_total
table["share_vap"]   = table["vap_count"]/vap_total
table["share_cvap"]  = table["cvap_count"]/cvap_total

table[["group",
       "share_total","share_vap","share_cvap",
       "total_count","vap_count","cvap_count"]].assign(
    share_total=lambda d: d["share_total"].map(pct),
    share_vap=lambda d: d["share_vap"].map(pct),
    share_cvap=lambda d: d["share_cvap"].map(pct),
)


,group,share_total,share_vap,share_cvap,total_count,vap_count,cvap_count
0,Latino,39.26%,36.16%,31.64%,11441717,7907319,6285665
1,Black,13.60%,12.98%,13.17%,3964700,2837841,2616672
2,White,39.75%,43.16%,48.28%,11584597,9437993,9592790
3,Other,1.22%,1.55%,2.19%,356472,339468,436016


In [137]:
float((table["cvap_count"].sum()/cvap_total))


0.9528426240114225

In [138]:
geo.shape
geo[["total_pop","vap_total","cvap_total"]].sum()
table[["group","share_total","share_vap","share_cvap"]].assign(
    share_total=lambda d: d["share_total"].map(pct),
    share_vap=lambda d: d["share_vap"].map(pct),
    share_cvap=lambda d: d["share_cvap"].map(pct),
)


,group,share_total,share_vap,share_cvap
0,Latino,39.26%,36.16%,31.64%
1,Black,13.60%,12.98%,13.17%
2,White,39.75%,43.16%,48.28%
3,Other,1.22%,1.55%,2.19%


In [139]:
(geo["cvap_total"] - geo["vap_total"]).describe()


count     9712.000000
mean      -205.789848
std        561.576253
min     -10080.000000
25%       -382.000000
50%        -85.500000
75%         22.000000
max       4183.000000
dtype: float64

In [140]:
# Total amount by which CVAP exceeds VAP statewide
excess = (geo["cvap_total"] - geo["vap_total"]).clip(lower=0).sum()
excess


np.int64(731001)

In [141]:
excess / geo["cvap_total"].sum()


np.float64(0.036792755249642026)

In [142]:
(geo["vap_total"] - geo["total_pop"]).describe()


count    9712.000000
mean     -749.465095
std       674.934014
min     -6891.000000
25%     -1112.250000
50%      -598.000000
75%      -213.000000
max         0.000000
dtype: float64

In [143]:
(geo["cvap_total"] > geo["vap_total"]).mean()


np.float64(0.29849670510708404)

In [144]:
(geo["cvap_total"] - geo["vap_total"]).clip(lower=0).describe()


count    9712.000000
mean       75.267813
std       219.247010
min         0.000000
25%         0.000000
50%         0.000000
75%        22.000000
max      4183.000000
dtype: float64

In [145]:
rel_excess = ((geo["cvap_total"] - geo["vap_total"])
              .clip(lower=0) / geo["vap_total"].replace(0, np.nan))

rel_excess.describe()


count    9381.000000
mean        0.065178
std         2.007250
min         0.000000
25%         0.000000
50%         0.000000
75%         0.032172
max       194.000000
dtype: float64

In [146]:
geo.loc[
    rel_excess > 5,  # greater than 500%
    ["vtd_geoid", "vap_total", "cvap_total"]
].sort_values("vap_total").head(20)


,vtd_geoid,vap_total,cvap_total
8470,8471.0,5,975
6583,6584.0,375,2975


In [147]:
(geo["vap_total"] < 50).sum()


np.int64(616)

In [148]:
mask = geo["vap_total"] >= 50
excess_large = ((geo.loc[mask, "cvap_total"]
                 - geo.loc[mask, "vap_total"])
                 .clip(lower=0).sum())

excess_large / geo.loc[mask, "cvap_total"].sum()


np.float64(0.036734405142428025)